# DAF-18: The Board Paper — Final Test, Error Analysis, and the Model Card

## 📖 The story: Asha madam opens the sealed envelope

### Who is Asha madam, and what must she decide?

Asha madam is the coordinator of a school in Delhi. Delhi's air is sometimes very bad. The tiny dust in the air is measured as **PM2.5**. A higher number means worse air, and **91 or more is "Poor"**. Children should not stand outside on a Poor day.

Every evening at 6 pm she must decide one thing:

> **Tomorrow morning, will the 600 children stand outside for assembly, or stay indoors?**

To decide, she needs a guess of **tomorrow's PM2.5**.

### Two ways she can guess

| | Way 1: her old method | Way 2: her model |
|---|---|---|
| Name | **Persistence** | **Ridge** (a small machine-learning model) |
| How it guesses | She looks at today's 5 pm reading and says, *"Tomorrow will be about the same."* | The computer looks at today's reading and the average of the last 3, 7 and 14 days, and predicts tomorrow's number |
| Cost | Free, no computer | Needs training, a file to save, a service to run |

The whole project asks one question: **is Way 2 really better than Way 1?** If it is not, why keep the computer?

### What she has done so far (a school comparison)

| Ticket | What happened | Like in school |
|---|---|---|
| DAF-14 to DAF-16 | She built the model and tried it | Learning the chapters and writing practice tests |
| DAF-17 | She tried the settings (the "knobs") using only the 251 **study days**. Then she wrote her final choice in `models/final_config.json` and **signed** it | Finishing preparation, writing the final study plan on paper, and signing it |
| **DAF-18 (now)** | She opens the **130 test days** (from 1 March 2026) | **The real board exam paper** |

Those 130 test days are the **sealed envelope**. The model has never been allowed to learn from them. That is why they can give an honest answer.

### The problem

The principal asks three plain questions:

> *"Does your model beat your old method? By how much? And on which days should I not trust it?"*

Asha madam is nervous. In the mock tests (DAF-17), her model won only a little: an average mistake of 36.5 against persistence's 37.9. That is a win of only about 1.4 out of about 37, which is far below the 10% the project asked for.

Now the envelope is on the table, and she feels a temptation:

```text
1. She opens the envelope and sees a poor score.
2. She quietly changes one setting.
3. She opens the envelope again. The score looks better.
4. She tells the principal the second score.
```

### What Sharma sir says

Sharma sir, the old maths teacher, stops her.

> *"Beta, the paper opens once. If the marks are poor, write the poor marks in the report card. A student who retakes the board paper until the marks look good has not passed. She has only learned the paper."*

**Why he is right.** The first score is the only honest one. The moment you change something because of what the test showed you, the test days have started to teach your model. They are no longer a fair test, like a student who has seen the question paper before the exam.

### So what do we do in this ticket?

```text
Open the envelope ONCE
   -> compare the model with persistence on the SAME 130 days
   -> say "met" or "not met" for each of the two pass marks
   -> open the answer sheet: which 10 days did we lose the most marks on, and why?
   -> write a one-page model card so that anyone can see where NOT to trust it
```

### One honest note

The test days were already looked at in earlier tickets (DAF-06 to DAF-16). So the envelope was not perfectly sealed. This is written down in decision D-007. It is why this notebook calls DAF-18 **the last look** at the test days, **not the first**, and why the model card must say it too.

## 🎯 The exact problem

> **How do we score the frozen model once, say honestly whether it met the goal, find the days where it fails, and write that down so that anyone who uses the model knows its limits?**

The answer is four things: **one test run**, **a verdict** against the two success rules, **the ten worst days**, and **a model card**.


## What this ticket is about

No new model and no new settings. This ticket only **measures and reports**.

```text
DAF-17 ended with                        DAF-18 ends with
a frozen recipe (Ridge, alpha=1.0)       the recipe trained on all 251 train days
CV scores on 251 train days              ONE score on the 130 test days
"probably a bit better than persistence" a verdict: met / not met, with numbers
no idea where it fails                   ten worst days, each with a reason
a JSON file                              a saved model + a model card
```

The success rules come from `docs/requirements.md`. Both must hold:

| Rule | Meaning |
|---|---|
| MAE at least **10% lower** than persistence | the average mistake is clearly smaller |
| Recall on Poor-or-worse days **not lower** than persistence | it must not miss more smoggy days than the old method |

Poor means PM2.5 of 91 µg/m³ or more.

## The new ideas

| Idea | One line | Where it comes from |
|---|---|---|
| Final test run | Fit on all train rows, predict the locked test rows, score them **once** | the frozen recipe in `models/final_config.json` |
| Verdict | A plain "met" or "not met" for each success rule, with the numbers next to it | `docs/requirements.md` |
| Error analysis | Sort the test days by how wrong we were and look at the worst ten | pandas `sort_values` |
| `joblib` | Save a fitted model to one file and load it again later, even in another Python session | `joblib.dump` / `joblib.load` |
| Model card | A one-page note that travels with the model: purpose, data, features, metrics, threshold, failure cases, do-not-use limits | `docs/model_card.md` |

DAF-16 (walk-forward) is reported next to the test score as a second opinion. It is **not** a second test.

## What we start with

Everything the model needs was decided in DAF-17 and is in `models/final_config.json`:

| Item | Frozen value |
|---|---|
| Model | `StandardScaler` then `Ridge(alpha=1.0, solver="lsqr")` |
| Features | `pm25_until_17`, `mean_3`, `mean_7`, `mean_14` |
| Train rows | 251 days, 17 Mar 2025 – 28 Feb 2026 |
| Locked test | 130 days, from 1 Mar 2026 |
| Poor line | 91 µg/m³ |
| Cross-validation (train days only) | Ridge 36.456 vs persistence 37.889 MAE, about **3.8% better** |

That last row is a warning to ourselves, written **before** we open the envelope: 3.8% is far below the 10% target. The test result may well say "not met". That is a valid outcome of this ticket.

## Rules for this ticket

- **Read the recipe from the file.** The model is rebuilt from `models/final_config.json`. Nothing is typed in again by hand.
- **One scoring.** The test rows are predicted once, in one cell. No second attempt, no "just checking" with a small change.
- **Write the verdict rule first.** The two success rules are copied into the notebook before the score is computed.
- **Persistence on the same rows.** The baseline is scored on exactly the same 130 days, otherwise the comparison is unfair.
- **Be honest about history.** The test days were already looked at in DAF-06 to DAF-16 (D-007 says so). This is the last look at them, not the first, and the model card must say that.
- **Report what happened.** If the model loses, the report says it loses.

## The plan, one step at a time

This is the proposed plan. Each step is added to this notebook only when we start it.

1. Rebuild the train and test tables from the frozen config. Check the fingerprint so we know the recipe and data are the ones DAF-17 froze.
2. Write the verdict rules in the notebook, before any score exists.
3. Train Ridge on all 251 train rows. Score the 130 test rows **once**, with persistence beside it.
4. Put the walk-forward result from DAF-16 next to the test result.
5. Verdict: criterion 1 (MAE) and criterion 2 (recall), each with its numbers.
6. Error analysis: the ten worst test days, with date, actual, forecast, persistence forecast, and a likely reason.
7. Save the model with `joblib` in `models/pm25_station17_v1.joblib`, with the feature list and training date range. Load it in a fresh Python process and check that it reproduces one test prediction.
8. Write `docs/model_card.md`.
9. Log the rows in `reports/experiments.csv`.

The pictures for each step go in `18_visual_walkthrough.ipynb`, the same way `17_visual_walkthrough.ipynb` went with DAF-17.

## Acceptance criteria (from the ticket)

- [ ] The test set is scored exactly once — the notebook shows no second attempt
- [ ] The verdict answers both success criteria with numbers
- [ ] The ten-worst-days table exists with reasons
- [ ] The model file loads in a fresh Python session and reproduces one test prediction
- [ ] `docs/model_card.md` has every section in step 5 of the ticket

## Explain back (answered at the end)

1. Did the model meet the success criteria? If not, what is the most likely reason, and what would you try next?
2. What would you tell Asha about when not to trust the forecast?